# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. `1 1.3333333333333333`. В Series один пропуск. После `fillna(0)` получается [1, 0, 3], а среднее 4 / 3.
2. `2`. После `strip` и `lower` и `" a"`, и `"A "` превращаются в `"a"`, остаётся два разных значения: `a` и `b`.
3. `3`. `drop_duplicates()` возвращает новую таблицу, но её никуда не сохранили, `df` остался прежним.
4. `1005`. Пробел убирается, `"1 000"` становится 1000, плюс 5.
5. `[9, 10]`. После `to_datetime` это настоящие даты, у первой месяц сентябрь, у второй октябрь.

In [1]:
# проверка предсказаний
import pandas as pd
import numpy as np

s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

1 1.3333333333333333
2
3
1005
[9, 10]


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [3]:
# 1. функция очистки
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

def clean(path):
    df = pd.read_csv(path)
    print("прочитали:", len(df))

    # суммы "15 990" -> 15990.0
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    assert df["amount"].isna().sum() == 0, "в amount есть NaN"
    print("суммы в числа:", len(df))

    # 14 написаний -> 6 городов
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    assert df["city"].isna().sum() == 0, "город не нашёлся в словаре"
    print("города через словарь:", len(df), "| городов:", df["city"].nunique())

    df = df.drop_duplicates()
    assert df["order_id"].is_unique, "order_id повторяется"
    print("без дублей:", len(df))

    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print("даты:", len(df), "|", df["order_date"].dtype)

    # отменённые не трогаем, они нужны для доли отмен
    df = df[df["customer_id"] != "TEST"].copy()
    print("без TEST:", len(df))
    return df

df = clean("orders.csv")
print("\nстатусы:", df["status"].value_counts().to_dict())

прочитали: 5463
суммы в числа: 5463
города через словарь: 5463 | городов: 6
без дублей: 5283
даты: 5283 | datetime64[us]
без TEST: 5280

статусы: {'оплачен': 4739, 'отменён': 541}


**Ответ:** после очистки осталось 5 280 строк. По пути ушли 180 повторов и 3 заказа TEST, а суммы, города и даты строк не убирают, только приводят к нормальному виду. Отменённые заказы на месте, их 541.

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [4]:
# 2. выручка оплаченных по месяцам, всего и по городам
paid = df["status"] == "оплачен"

def change(mask):
    sep = df.loc[mask & (df["month"] == 9), "amount"].sum()
    octo = df.loc[mask & (df["month"] == 10), "amount"].sum()
    return sep, octo, (octo / sep - 1) * 100

s, o, ch = change(paid)
print(f"всего: сентябрь {s:,.0f} ₸, октябрь {o:,.0f} ₸, изменение {ch:.1f}%\n")

for c in sorted(df["city"].unique()):
    s, o, ch = change(paid & (df["city"] == c))
    print(f"{c:<10} {s:>12,.0f} {o:>12,.0f}  {ch:+.1f}%")

всего: сентябрь 35,243,410 ₸, октябрь 34,419,520 ₸, изменение -2.3%

Актобе        2,576,960    2,540,420  -1.4%
Алматы       14,501,070   14,643,020  +1.0%
Астана        9,034,220    8,850,530  -2.0%
Атырау          923,990      969,860  +5.0%
Караганда     3,648,520    3,371,680  -7.6%
Шымкент       4,558,650    4,044,010  -11.3%


**Ответ:** по чистым данным выручка 35 243 410 ₸ в сентябре и 34 419 520 ₸ в октябре, падение 2,3%. Суммы по всем шести городам до тенге совпадают с таблицей городов по месяцам из урока 4.

Сильнее всех падает Шымкент, -11,3%, потом Караганда, -7,6%. Алматы и Атырау наоборот немного выросли.

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [5]:
# 3. как менялась цифра менеджера по ходу очистки
raw = pd.read_csv("orders.csv")
raw["amount"] = pd.to_numeric(raw["amount"].str.replace(" ", ""))

def drop_pct(t):
    p = t["status"] == "оплачен"
    sep = t["order_date"] < "2026-10"   # тут даты ещё текст
    s, o = t.loc[p & sep, "amount"].sum(), t.loc[p & ~sep, "amount"].sum()
    return (o / s - 1) * 100

no_dup = raw.drop_duplicates()
no_test = no_dup[no_dup["customer_id"] != "TEST"]
print(f"как считал менеджер: строк {len(raw)}, {drop_pct(raw):.1f}%")
print(f"без дублей:          строк {len(no_dup)}, {drop_pct(no_dup):.1f}%")
print(f"без TEST:            строк {len(no_test)}, {drop_pct(no_test):.1f}%")

# если убирать в другом порядке
only_test = raw[raw["customer_id"] != "TEST"]
print(f"\nсначала TEST: {drop_pct(only_test):.1f}%, потом дубли: {drop_pct(no_test):.1f}%")
dup = raw[raw.duplicated()]
print(f"выручка оплаченных дублей: {dup.loc[dup['status'] == 'оплачен', 'amount'].sum():,.0f} ₸")

как считал менеджер: строк 5463, -22.7%
без дублей:          строк 5283, -19.6%
без TEST:            строк 5280, -2.3%

сначала TEST: -8.2%, потом дубли: -2.3%
выручка оплаченных дублей: 2,246,460 ₸


**Журнал очистки:**

| Шаг | Убрано строк | Осталось | Падение выручки окт/сент |
|---|---|---|---|
| Как считал менеджер | 0 | 5 463 | -22,7% |
| Суммы из текста в числа | 0 | 5 463 | -22,7% |
| Города к 6 названиям | 0 | 5 463 | -22,7% |
| Убрали дубли | 180 | 5 283 | -19,6% |
| Даты в datetime | 0 | 5 283 | -19,6% |
| Убрали TEST | 3 | 5 280 | -2,3% |

Суммы, города и даты цифру не меняют, они нужны, чтобы её вообще можно было правильно посчитать. Если убирать в другом порядке, сначала TEST, получается -22,7% → -8,2% → -2,3%.

**Менеджеру:** сильнее всего цифру исказил тестовый клиент TEST. У него всего 3 заказа по 10 млн ₸, но два из них попали в сентябрь и раздули его, отсюда и страшные -22,7%. Повторы строк добавили к сентябрю ещё около 2,2 млн ₸.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [6]:
# 4. пропуски в сроках доставки
no_days = df["delivery_days"].isna()
print("пропусков:", no_days.sum(), "из", len(df))
print("\nпо городам:")
print(df.loc[no_days, "city"].value_counts())
print("\nпо статусам:")
print(df.loc[no_days, "status"].value_counts())

# Шымкент: доля доставки за 2 дня тремя способами
d = df.loc[df["city"] == "Шымкент", "delivery_days"]
print(f"\nШымкент: заказов {len(d)}, без срока {d.isna().sum()}, медиана {d.median():.0f} дн.")
print(f"(а) dropna:          {(d.dropna() <= 2).mean():.1%}")
print(f"(б) fillna(медиана): {(d.fillna(d.median()) <= 2).mean():.1%}")
print(f"(в) fillna(0):       {(d.fillna(0) <= 2).mean():.1%}")

пропусков: 158 из 5280

по городам:
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64

по статусам:
status
оплачен    144
отменён     14
Name: count, dtype: int64

Шымкент: заказов 776, без срока 22, медиана 5 дн.
(а) dropna:          29.8%
(б) fillna(медиана): 29.0%
(в) fillna(0):       31.8%


**Ответ:** в чистых данных 158 заказов без срока доставки (в исходном файле было 160, два ушли вместе с дублями). Больше всего их в Алматы (63) и Астане (40), но это просто самые большие города. В процентах сильнее выделяется Караганда, там без срока около 4,6% заказов. По статусам: 144 оплаченных и 14 отменённых.

Шымкент, доставка за 2 дня: dropna 29,8%, fillna(медиана) 29,0%, fillna(0) 31,8%.

Честный вариант dropna: считаем только по заказам, где срок известен, и ничего не придумываем. С медианой мы как будто знаем, что все 22 заказа шли 5 дней, и доля немного занижается. С нулём ещё хуже: 0 дней быстрее любой реальной доставки, и Шымкент выглядит лучше, чем есть.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [7]:
# 5. код из условия: 10 городов и 5 463 строки
bad = pd.read_csv("orders.csv")
bad.drop_duplicates()
bad["city"] = bad["city"].str.lower()
print("было:", bad["city"].nunique(), len(bad))
print(sorted(bad["city"].unique()))

# ошибка 1: drop_duplicates без присваивания, таблица не меняется
# ошибка 2: только lower, без strip и без словаря для латиницы
fixed = pd.read_csv("orders.csv")
fixed = fixed.drop_duplicates()
fixed["city"] = fixed["city"].str.strip().str.lower().map(CITY)
print("стало:", fixed["city"].nunique(), len(fixed))

было: 10 5463
[' алматы', 'almaty', 'astana ', 'shymkent', 'актобе', 'алматы', 'астана', 'атырау', 'караганда', 'шымкент']
стало: 6 5283


**Ответ:** первая ошибка: `drop_duplicates()` не присвоили обратно в `df`, поэтому строк осталось 5 463 вместо 5 283. Вторая: город только переводится в нижний регистр. Пробелы по краям остаются (`" алматы"`, `"astana "`), латиница тоже (`almaty`, `shymkent`), отсюда 10 значений. После `df = df.drop_duplicates()` и `.str.strip().str.lower().map(CITY)` получается 6 городов и 5 283 строки.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [8]:
# 6. заказы по дням недели
RU = {"Monday": "пн", "Tuesday": "вт", "Wednesday": "ср", "Thursday": "чт",
      "Friday": "пт", "Saturday": "сб", "Sunday": "вс"}
by_day = df["order_date"].dt.day_name().map(RU).value_counts()
print(by_day)
print(f"\nменьше всего: {by_day.idxmin()} {by_day.min()}, больше всего: {by_day.idxmax()} {by_day.max()}, "
      f"разница {(1 - by_day.min() / by_day.max()) * 100:.0f}%")

# но самих дней недели в сентябре и октябре разное количество
dates = pd.to_datetime(df["order_date"].dt.date.drop_duplicates())
n_days = dates.dt.day_name().map(RU).value_counts()
print("\nсколько раз был каждый день:", n_days.to_dict())

per_day = (by_day / n_days).sort_values()   # делится по ярлыкам, как на уроке 5
print("\nзаказов в среднем за один день:")
print(per_day.round(1))
print(f"разница между крайними днями: {(1 - per_day.min() / per_day.max()) * 100:.1f}%")

order_date
чт    786
вт    785
пт    781
ср    774
сб    773
вс    697
пн    684
Name: count, dtype: int64

меньше всего: пн 684, больше всего: чт 786, разница 13%

сколько раз был каждый день: {'вт': 9, 'ср': 9, 'чт': 9, 'пт': 9, 'сб': 9, 'вс': 8, 'пн': 8}

заказов в среднем за один день:
order_date
пн    85.5
сб    85.9
ср    86.0
пт    86.8
вс    87.1
вт    87.2
чт    87.3
Name: count, dtype: float64
разница между крайними днями: 2.1%


**Ответ:** если просто посчитать заказы, меньше всего их в понедельник (684), больше всего в четверг (786), разница 13%. Но в сентябре и октябре было всего 8 понедельников и 8 воскресений, а остальных дней по 9. В пересчёте на один день картина ровная: от 85,5 заказа в понедельник до 87,3 в четверг, разница около 2%.

**Менеджеру:** нагрузка по дням недели почти одинаковая, понедельник только кажется тихим из-за календаря. Сокращать курьеров в какой-то день по этим данным не стоит, нужно примерно одинаковое число смен всю неделю.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [9]:
# 7. когда сломалась доставка в Шымкенте
sh = df[df["city"] == "Шымкент"]
for m, name in [(9, "сентябрь"), (10, "октябрь")]:
    print(f"{name}: средний срок {sh.loc[sh['month'] == m, 'delivery_days'].mean():.2f} дн.")

known = sh[~sh["delivery_days"].isna()].sort_values("order_date")
last_fast = known.loc[known["delivery_days"] < 5, "order_date"].max()
after = known[known["order_date"] > last_fast]
print("\nпоследний заказ быстрее 5 дней:", last_fast)
print("первый заказ после него:", after["order_date"].min(), "| срок:", after["delivery_days"].iloc[0])
print("после этого быстрее 5 дней было:", (after["delivery_days"] < 5).sum(), "заказов")

# остальные города за неделю до и после 1 октября
other = df[df["city"] != "Шымкент"]
week_before = other[(other["order_date"] >= "2026-09-24") & (other["order_date"] < "2026-10-01")]
week_after = other[(other["order_date"] >= "2026-10-01") & (other["order_date"] < "2026-10-08")]
print(f"\nдругие города: неделя до {week_before['delivery_days'].mean():.2f} дн., "
      f"неделя после {week_after['delivery_days'].mean():.2f} дн.")

for m, name in [(9, "сентябрь"), (10, "октябрь")]:
    part = sh[sh["month"] == m]
    print(f"отмены в Шымкенте, {name}: {(part['status'] == 'отменён').mean():.1%}")

сентябрь: средний срок 2.03 дн.
октябрь: средний срок 6.54 дн.

последний заказ быстрее 5 дней: 2026-09-30 22:00:00
первый заказ после него: 2026-10-01 10:00:00 | срок: 5.0
после этого быстрее 5 дней было: 0 заказов

другие города: неделя до 1.96 дн., неделя после 2.02 дн.
отмены в Шымкенте, сентябрь: 8.9%
отмены в Шымкенте, октябрь: 30.7%


**Ответ:** в сентябре заказы в Шымкенте доставляли в среднем за 2 дня, как везде, а в октябре уже за 6,5 дня. Последний заказ быстрее 5 дней был 30.09 в 22:00, а начиная с первого октябрьского заказа (01.10 в 10:00) срок больше не бывает меньше 5 дней. В других городах в эту дату ничего не поменялось: и до, и после около 2 дней.

**Менеджеру:** с 1 октября в Шымкенте сломалась доставка. Заказы едут 5-8 дней вместо 2, и отмен стало втрое больше (8,9% в сентябре, 30,7% в октябре). Отсюда и большая часть падения выручки. В первую очередь надо узнать, что поменялось в Шымкенте с 1 октября: курьерская служба, склад или подрядчик.

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [10]:
# 8. надёжный перевод сумм
def to_amount(s):
    s = s.astype(str).str.replace(" ", "").str.replace(",", ".")
    # coerce здесь нарочно: всё, что не число, станет NaN, и мы это проверяем ниже
    return pd.to_numeric(s, errors="coerce")

test = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
print(to_amount(test))

amounts = to_amount(pd.read_csv("orders.csv")["amount"])
print(f"\norders.csv: NaN {amounts.isna().sum()}, сумма {amounts.sum():,.0f} ₸")

0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64

orders.csv: NaN 0, сумма 109,883,360 ₸


**Ответ:** функция убирает пробелы, меняет запятую на точку и переводит в число, а всё, что числом не стало, превращает в NaN. На примере получилось 15990, 15990, 1200.5 и NaN. На столбце `amount` из orders.csv ни одного NaN, а сумма 109 883 360 ₸, ровно как на уроке.